# Belgian Real-Estate — Data Analysis

**BeCode project — data cleaning, exploration and analysis.**

This notebook turns the raw scraped dataset into clean, analysable data and answers two questions.
The logic is organised in **three classes**, one responsibility each:

- **`DataCleaner`** — loads the raw CSV and returns a cleaned DataFrame.
- **`DataVisualizer`** — general overview charts (dashboard + price map).
- **`MeetingAnswers`** — the two meeting questions, each with a chart:
  - **Q1.** Which variables would you delete, and why?
  - **Q2.** Which five variables most influence the price, and why?

## <span style="color:#c8a2c8; font-weight:bold">Irene's changes on Dan's DataCleaner class</span>

## 1. Setup

Imports, plotting style, and the path to the dataset.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

from IPython.display import display, HTML

# forcing pandas to display whole url
pd.set_option('display.max_colwidth', None)

# pforcing pandas to display all the columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

%matplotlib inline
sns.set_theme(style="whitegrid")

# Put properties_cleaned.csv next to this notebook, or edit the path below.
CSV_PATH = "../../data/raw/properties_parsed.csv"

## 2. Data cleaning

`DataCleaner` applies an 8-step pipeline:

1. **Drop useless columns** — `price_type` (constant) and `property_id` (duplicate of `property_url`).
2. **Normalise the city** — `la-roche-en-ardenne` → `La Roche En Ardenne`.
3. **Standardise the building state** — merge synonyms (`To be renovated` → `To renovate`) + *ordered* category.
4. **Ordered categories** for EPC and kitchen.
5. **Booleans → nullable `Int8`** (with an explicit assumption on the `has_*` flags).
6. **Fix swapped coordinates** — only when the swap is *confirmed by the postal code* (an independent source).
7. **Flag suspect values** — `price_suspect`, `area_suspect`, `year_suspect` (without modifying them).
8. **Derived feature** — `price_per_m2`.

> Principle: auto-correct only what is **indisputable** (geographic swaps verified against the postal code),
> and merely **flag** what needs judgment (price/area/year outliers).

In [2]:
class DataCleaner:
    """Loads the raw scraper CSV and produces a cleaned DataFrame.

    All cleaning knowledge (category orders, synonyms, geographic bounds,
    plausibility thresholds) lives here as class attributes, so there is a
    single place to update.
    """

    BOOL_COLS = [
        "furnished", "has_garage", "has_garden", "has_terrace",
        "has_elevator", "is_nearby_city_prestigious",
    ]
    PRESENCE_FLAGS = ["has_garage", "has_garden", "has_terrace", "has_elevator", "furnished"]

    EPC_ORDER = ["A++", "A+", "A", "B+", "B", "C", "D", "E+", "E", "F", "G"]
    STATE_ORDER = [
        "New", "Excellent", "Fully renovated", "Normal",
        "To renovate", "To restore", "Under construction", "To demolish",
    ]
    STATE_SYNONYMS = {"To be renovated": "To renovate"}
    KITCHEN_ORDER = ["Not equipped", "Partially equipped", "Fully equipped", "Super equipped"]

    # Geographic bounds of Belgium + max distance to accept a coordinate swap.
    BE_LAT = (49, 52)
    BE_LON = (2, 7)
    SWAP_MAX_KM = 25

    # Bedroom/area plausibility FLAG (review, not deletion). No threshold: every
    # listing is checked against a loose per-bedroom minimum.
    MIN_M2_PER_BEDROOM_SMALL = 9
    MIN_COMMON_AREA_M2 = 15

    # Hard floors for DELETION — physically impossible values only.
    HARD_MIN_PRICE = 19_900        # no real residential sale below this
    HARD_MIN_AREA = 10             # m², below this a dwelling is impossible
    HARD_MIN_M2_PER_BEDROOM = 5    # absolute floor: a bedroom needs at least this
    # new class constant for managing minimum price (HARD_MIN_PRICE updated) and maximum building year
    MAX_BUILDING_YEAR = 2026

    # urls to drop set
    URLS_TO_DROP = {
        "https://immovlan.be/en/real-estate/house/for-sale/sint-truiden",
        "https://immovlan.be/en/detail/master-house/for-sale/3540/herk-de-stad/rbv60505"
    }


    def __init__(self, path):
        """Store the path and load the raw CSV (kept untouched in self.raw)."""
        self.path = path
        self.raw = pd.read_csv(path)   # untouched copy, useful for the Q1 audit
        self.df = None                 # filled by clean()

    # ----- geographic helpers -----

    @staticmethod
    def _haversine(lat1, lon1, lat2, lon2):
        """Great-circle distance in km between two points (decimal degrees)."""
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
        return 2 * R * np.arcsin(np.sqrt(a))

    def _postal_centroids(self, df):
        """Median lat/lon per postal code, from rows already inside Belgium.

        Independent ground truth (postal code comes from the URL) used to
        validate coordinate swaps.
        """
        valid = df["latitude"].between(*self.__class__.BE_LAT) & df["longitude"].between(*self.__class__.BE_LON)
        return df[valid].groupby("postal_code")[["latitude", "longitude"]].median()

    def _fix_swapped_coordinates(self, df):
        """Swap lat/lon only when it brings the point near its postal centroid.

        Rows where the swap is not confirmed are left untouched and flagged
        coord_suspect. Adds coord_swapped and coord_suspect.
        """
        cent = self._postal_centroids(df)

        # setting conditions for candidate
        
        lan_lon_present = df["latitude"].notna() & df["longitude"].notna()
        #   checks if latitude and longitude are not nan 
        lan_lon_out_of_be = ~ (df["latitude"] .between(*self.__class__.BE_LAT) & df["longitude"].between(*self.__class__.BE_LON))
        #   checks if latitude and longitude are not outside of Belgium

        candidate = (
            (lan_lon_present) &
            (lan_lon_out_of_be)
        )

        df["coord_swapped"] = False
        df["coord_suspect"] = False

        # CODE VECTORIALIZATION
        # substitution of the original for cycle

        # Map centroids by postal code to avoid a heavy merge
        clat = df['postal_code'].map(cent['latitude'])
        clon = df['postal_code'].map(cent['longitude'])

        # calling _haversine method but switching original long and lan
        distance_with_cent = self._haversine(
            df['longitude'], 
            df['latitude'], 
            clat, 
            clon)
        
        # cond_masq immediatelly identifies rows that pass the controls
        cond_masq = (
            (candidate) &
            (distance_with_cent < self.__class__.SWAP_MAX_KM)
        )

        # setting True in 'coord_swapped' column only for row identified by cond_masq
        df.loc[cond_masq, "coord_swapped"] = True

        # new condition masq for suspect coordinates
        suspect_masq = (
            (candidate) &
            ~ (cond_masq)
        )

        # setting True in 'coord_suspect' column in rows where:
        #       is a candidate
        #       _havesine() fails
        df.loc[suspect_masq, "coord_suspect"] = True

        # Vectorized lat/lon swap for confirmed rows
        df.loc[cond_masq, ['longitude', 'latitude']] = df.loc[cond_masq, ['latitude', 'longitude']].values

        return df

    # ----- plausibility flag + deletion -----

    def _flag_suspect_bedroom_count(self, df):
        """Flag listings whose living area is too small for their bedroom count.

        No threshold: every listing is checked against a loose per-bedroom
        minimum (MIN_M2_PER_BEDROOM_SMALL each + MIN_COMMON_AREA_M2). This is a
        REVIEW flag, not a deletion rule — small but legitimate studios may be
        flagged. Adds bedroom_suspect (bool); does not modify any value.
        """
        min_required_area = (
            df["bedrooms"] * self.__class__.MIN_M2_PER_BEDROOM_SMALL + self.__class__.MIN_COMMON_AREA_M2
        )
        df["bedroom_suspect"] = df["living_area_m2"] < min_required_area
        return df

    def _drop_impossible_rows(self, df):
        """Delete physically impossible rows (real errors, not judgment calls).

        Removes listings whose price or area cannot match a real dwelling: price
        below HARD_MIN_PRICE, living area below HARD_MIN_AREA, or living area too
        small for the bedroom count. Prints how many rows were removed.
        (NaN comparisons return False, so rows with missing values are NOT dropped.)
        """
        # commented code below was originally in clean() method, now optimized in here
        # to_drop = (df["price"] < 19_900) | (df["building_year"] >= 2027)
        # df = df[~to_drop].copy()

        before = len(df)

        # updating impossible_masq with building year condition
        impossible_masq = (
            (df['price'].isna()) |
            (df["price"] < self.__class__.HARD_MIN_PRICE) |
            (df["living_area_m2"] < self.__class__.HARD_MIN_AREA) |
            (df["living_area_m2"] < df["bedrooms"] * self.__class__.HARD_MIN_M2_PER_BEDROOM) |
            (df["building_year"] > self.__class__.MAX_BUILDING_YEAR)
        )

        df = df[~impossible_masq].copy()

        print(f"Dropped {before - len(df)} impossible rows "
              f"(price < {self.__class__.HARD_MIN_PRICE} EUR, area < {self.__class__.HARD_MIN_AREA} m2, "
              f"or < {self.__class__.HARD_MIN_M2_PER_BEDROOM} m2/bedroom)")

        return df

    # ----- main entry -----

    def clean(self):
        """Run the full cleaning pipeline and return the cleaned DataFrame."""
        df = self.raw.copy()

        # 1) Drop useless columns (constant + duplicate). See Q1.
        df = df.drop(columns=[c for c in ["price_type", "property_id"] if c in df.columns])

        # 2) Normalise the city slug: "la-roche-en-ardenne" -> "La Roche En Ardenne".
        df["city"] = df["city"].str.replace("-", " ", regex=False).str.title()

        # 3) Building state: merge synonyms + ordered category.
        df["state_of_the_building"] = df["state_of_the_building"].replace(self.__class__.STATE_SYNONYMS)
        df["state_of_the_building"] = pd.Categorical(
            df["state_of_the_building"], categories=self.__class__.STATE_ORDER, ordered=True
        )

        # 4) EPC and kitchen as ordered categories (values outside the list -> NaN).
        df["epc_score"] = pd.Categorical(df["epc_score"], categories=self.__class__.EPC_ORDER, ordered=True)
        df["kitchen_equipped"] = pd.Categorical(
            df["kitchen_equipped"], 
            categories=self.__class__.KITCHEN_ORDER, 
            ordered=True
        )

        # 5) Booleans -> nullable Int8. NOTE: has_* only recorded "Yes" (1), so a
        #    NaN means "not mentioned"; NaN->0 is a modelling CHOICE.
        
        # CODE VECTORIALIZATION
        # substitution of original for cycle

        # action on missing values in PRESENCE_FLAGS columns
        df[self.__class__.PRESENCE_FLAGS] = df[self.__class__.PRESENCE_FLAGS].fillna(0)

        # action on data types of values in BOOL_COLS columns
        df[self.__class__.BOOL_COLS] = df[self.__class__.BOOL_COLS].astype('Int8')

        # 6) Fix swapped coordinates (validated against the postal code).
        df = self._fix_swapped_coordinates(df)
        
        # 6.5) Repair coord_suspect rows that have a trusted postal-code centroid.
        cent = self._postal_centroids(df)

        mask = df["coord_suspect"] & df["postal_code"].isin(cent.index)
        df.loc[mask, "latitude"]  = df.loc[mask, "postal_code"].map(cent["latitude"])
        df.loc[mask, "longitude"] = df.loc[mask, "postal_code"].map(cent["longitude"])
       
        # 7) delete wrong data

        # FIXING HARDCODING 
        # set 'URLS_TO_DROP' created as class attribute

        # generating Boolean masq (True for urls in URL_TO_DROP)
        url_to_drop_masq = ~ df["property_url"].isin(self.__class__.URLS_TO_DROP)
        # assigning Boolean masq to df (rows were condition is True are deleted)
        df = df[url_to_drop_masq]
        
        df = self._flag_suspect_bedroom_count(df)

        # 7.5) former 9 (now calculating the "price_per_m2" after removal of critical data)
        # Delete physically impossible rows (logged). The flags above keep the
        #    "suspect-but-plausible" cases for review; here we only remove errors.
        df = self._drop_impossible_rows(df)
        df = df.reset_index(drop=True)   # tidy index after the deletion

        # 8) Derived feature: price per m².
        df["price_per_m2"] = (df["price"] / df["living_area_m2"]).round(0)

        # 9) 
        # code has been moved before price_per_m2 calculation (7.5)
        
        # ADDING STEP 10 IN CLEANING
        # 10) removing 'garbage' columns from final df
        # df["coord_suspect"] from _fix_swapped_coordinates(df)
        # df["bedroom_suspect"] from _flag_suspect_bedroom_count(df)
        df = df.drop(columns=["coord_suspect", "bedroom_suspect"], errors="ignore")

        self.df = df
        return df

### ***<span style="color:#c8a2c8">IRENE: for clarification purposes, I renamed properties_cleaned.csv to properties_parsed.csv</span>***

### *<span style="color:#c8a2c8; font-weight:bold">IRENE: deleted columns "coord_suspect" and "bedroom_suspect", not useful to data analysis</span>*

##### *<span style="color:#c8a2c8">IRENE: initiate the cleaner and run the pipeline + saving new dataset as properties_final_irene.csv</span>*

In [3]:
# paths configuration
CSV_PATH = "../../data/raw/properties_parsed.csv"
OUTPUT_PATH = "../../data/cleaned/properties_final_irene.csv"

# pipeline initialization and execution
cleaner = DataCleaner(CSV_PATH)
df = cleaner.clean()

# Saving cleaned file
df.to_csv(OUTPUT_PATH, index=False)

# controls log
print(f"Cleaned: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Coordinates swapped: {int(df['coord_swapped'].sum())}")
print(f"Dataset successfully saved in: {OUTPUT_PATH}")
df.head()

df.describe(include='all')

Dropped 291 impossible rows (price < 19900 EUR, area < 10 m2, or < 5 m2/bedroom)
Cleaned: 12399 rows x 34 columns
Coordinates swapped: 44
Dataset successfully saved in: ../../data/cleaned/properties_final_irene.csv


,property_type,property_subtype,price,living_area_m2,bedrooms,bathrooms,address,postal_code,city,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,nearby_city,km_from_nearby_city,is_nearby_city_prestigious,floor_number,property_url,coord_swapped,price_per_m2
count,12399,12399,1.239900e+04,11557.000000,11987.000000,11036.000000,9949,12399.000000,12399,12397.000000,12397.000000,7433.000000,9541,12399.0,12399.0,4222.000000,3375,12399.0,9375.00000,5581.000000,12399.0,2780.000000,12399.0,6892.000000,10445,12399,12399,12397,12397.000000,12397.0,3508.000000,12399,12399,11557.000000
unique,2,15,NaN,NaN,NaN,NaN,9378,NaN,1601,NaN,NaN,NaN,8,<NA>,<NA>,NaN,4,<NA>,NaN,NaN,<NA>,NaN,<NA>,NaN,11,3,11,44,NaN,<NA>,NaN,12390,2,NaN
top,House,residence,NaN,NaN,NaN,NaN,Dessauerplein 1,NaN,Liege,NaN,NaN,NaN,Normal,<NA>,<NA>,NaN,Fully equipped,<NA>,NaN,NaN,<NA>,NaN,<NA>,NaN,C,Wallonia,Liège,Bruxelles,NaN,<NA>,NaN,https://immovlan.be/en/detail/residence/for-sale/7080/noirchain/vbe33505,False,NaN
freq,8023,6506,NaN,NaN,NaN,NaN,33,NaN,267,NaN,NaN,NaN,3665,<NA>,<NA>,NaN,1604,<NA>,NaN,NaN,<NA>,NaN,<NA>,NaN,2093,5988,1722,1386,NaN,<NA>,NaN,2,12355,NaN
mean,NaN,NaN,3.994536e+05,173.569525,3.078335,1.411290,NaN,5014.645294,NaN,50.720212,4.548858,1970.027849,NaN,0.036051,0.451246,1.860966,NaN,0.193241,2.95904,3.313922,0.536817,1232.853237,0.658279,1560.218659,NaN,NaN,NaN,NaN,35.256344,0.062999,3.541334,NaN,NaN,2579.397854
std,NaN,NaN,3.708000e+05,156.447090,2.275741,1.168308,NaN,2678.774296,NaN,0.371949,0.816695,47.241161,NaN,0.186425,0.497637,29.254179,NaN,0.394857,0.87970,2.936057,0.498663,5413.624467,0.474306,5068.651525,NaN,NaN,NaN,NaN,388.588079,0.242971,29.262686,NaN,NaN,1397.459709
min,NaN,NaN,1.990000e+04,12.000000,1.000000,1.000000,NaN,1000.000000,NaN,49.506790,2.509162,1500.000000,NaN,0.0,0.0,1.000000,NaN,0.0,1.00000,1.000000,0.0,1.000000,0.0,1.000000,NaN,NaN,NaN,NaN,0.000000,0.0,0.000000,NaN,NaN,106.000000
25%,NaN,NaN,1.950000e+05,90.000000,2.000000,1.000000,NaN,2570.000000,NaN,50.471983,4.155752,1950.000000,NaN,0.0,0.0,1.000000,NaN,0.0,2.00000,2.000000,0.0,117.000000,0.0,267.750000,NaN,NaN,NaN,NaN,3.200000,0.0,1.000000,NaN,NaN,1702.000000
50%,NaN,NaN,3.025000e+05,145.000000,3.000000,1.000000,NaN,4960.000000,NaN,50.771942,4.462025,1975.000000,NaN,0.0,0.0,1.000000,NaN,0.0,3.00000,3.000000,1.0,352.500000,1.0,629.000000,NaN,NaN,NaN,NaN,8.400000,0.0,2.000000,NaN,NaN,2400.000000
75%,NaN,NaN,4.990000e+05,207.000000,4.000000,2.000000,NaN,7100.000000,NaN,51.009126,5.237860,2007.000000,NaN,0.0,1.0,1.000000,NaN,0.0,4.00000,4.000000,1.0,900.000000,1.0,1316.250000,NaN,NaN,NaN,NaN,17.000000,0.0,3.000000,NaN,NaN,3158.000000


#### **Property types check**

In [4]:
property_types = df['property_type'].unique()

print("="*30 + " Property Types " + "="*30)
print(f"Property types in the dataset are {len(property_types)}: {', '.join(property_types)}")

============================== Property Types ==============================
Property types in the dataset are 2: House, Apartment


# deprecated
#### **Study of urban density based on postal code**  
Properties with postal codes linked to high density areas are more likely to not have a private parking spot.

In [ ]:
postal_codes = df['postal_code'].unique()

print("="*30 + " Postal Codes " + "="*30)
print(f"Postal codes in the dataset are {len(postal_codes)} in total.")

def categorize_area(df):
    """
    categorizes postal codes in:
        - 'High Density (City Center/Brussels)';
        - 'High Density (Urban Hub)';
        - 'Medium/Low Density'
    returns df with addictional column 'urban_density'.
    """
    pc_masq = [
        (df['postal_code'].between(1000, 1299)),            # Brussels Metropolitan Area
        (df['postal_code'] % 100 == 0)                      # other Provincial Urban Hubs (ending with 00)
    ]
    target_categories = ['High Density (City Center/Brussels)', 'High Density (Urban Hub)']
    # adding 'urban_density' to dataframe
    df['urban_density'] = np.select(pc_masq, target_categories, default='Medium/Low Density')
    return df

def high_urban_density_pc(df):
    """
    selects and returns postal codes which have:
        - 'High Density (City Center/Brussels)';
        - 'High Density (Urban Hub)'
    as value in 'urban_density' column.
    """
    df = categorize_area(df)
    high_urban_density_pc = df[
        (df['urban_density'] == 'High Density (City Center/Brussels)') |
        (df['urban_density'] == 'High Density (Urban Hub)')
    ]['postal_code'].unique()
    return high_urban_density_pc

print(
    f"Postal codes of total High Density Hubs in the dataset are {len(high_urban_density_pc(df))}:\n"
    f"{', '.join(map(str, high_urban_density_pc(df)))}"
)

df.to_csv(OUTPUT_PATH, index=False)
print(f"New version of dataset has been successfully saved in {OUTPUT_PATH}.")

We did not delete any row — outliers are only **flagged**, ready to be investigated later:

## 3. General visualization

An overview dashboard and a geographic price map.

In [ ]:
class DataVisualizer:
    """General overview charts for the cleaned dataset (rendered inline)."""

    def __init__(self, df):
        self.df = df

    def dashboard(self):
        """2x3 overview: prices, regions, types, area, EPC, provinces."""
        df = self.df
        fig, axes = plt.subplots(2, 3, figsize=(20, 11))
        fig.suptitle("Real-estate dataset — overview", fontsize=18, fontweight="bold")

        sns.histplot(df["price"].dropna(), bins=60, log_scale=True, ax=axes[0, 0], color="#4C72B0")
        axes[0, 0].set_title("Price distribution (log scale)")
        axes[0, 0].set_xlabel("Price (EUR)")

        sns.boxplot(data=df, x="region", y="price", ax=axes[0, 1], showfliers=False)
        axes[0, 1].set_title("Price by region")
        axes[0, 1].set_ylabel("Price (EUR)")

        df["property_type"].value_counts().plot(kind="bar", ax=axes[0, 2], color=["#55A868", "#C44E52"])
        axes[0, 2].set_title("Property type")
        axes[0, 2].set_xlabel("")
        axes[0, 2].tick_params(axis="x", rotation=0)

        sample = df.dropna(subset=["living_area_m2", "price"]).sample(
            min(3000, df["price"].notna().sum()), random_state=0
        )
        sns.scatterplot(data=sample, x="living_area_m2", y="price", hue="property_type",
                        alpha=0.4, s=18, ax=axes[1, 0])
        axes[1, 0].set_title("Living area vs price")
        axes[1, 0].set_xlabel("Living area (m2)")
        axes[1, 0].set_ylabel("Price (EUR)")

        order = [c for c in DataCleaner.EPC_ORDER if c in df["epc_score"].cat.categories]
        sns.countplot(data=df, x="epc_score", order=order, ax=axes[1, 1], color="#8172B3")
        axes[1, 1].set_title("Energy classes (EPC)")
        axes[1, 1].set_xlabel("EPC class")

        df["province"].value_counts().plot(kind="barh", ax=axes[1, 2], color="#CCB974")
        axes[1, 2].set_title("Number of listings per province")
        axes[1, 2].invert_yaxis()

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

    def price_map(self):
        """Geographic scatter of price per m2 across Belgium."""
        geo = self.df.dropna(subset=["latitude", "longitude", "price_per_m2"])
        lo, hi = geo["price_per_m2"].quantile([0.01, 0.99])   # clip for colour scale
        geo = geo[(geo["price_per_m2"] >= lo) & (geo["price_per_m2"] <= hi)]

        plt.figure(figsize=(11, 11))
        sc = plt.scatter(geo["longitude"], geo["latitude"], c=geo["price_per_m2"],
                         cmap="viridis", s=6, alpha=0.6)
        plt.colorbar(sc, label="Price per m2 (EUR)")
        plt.title("Price per m2 map (Belgium)", fontsize=15, fontweight="bold")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.gca().set_aspect("equal", adjustable="datalim")
        plt.show()

In [ ]:
viz = DataVisualizer(df)
viz.dashboard()

In [ ]:
viz.price_map()

## 4. Meeting questions

In [ ]:
class MeetingAnswers:
    """
    Q1 uses the RAW data (to justify deletions on the original columns);
    Q2 uses the CLEANED data (to rank what drives the price).
    """

    NON_FEATURES = [
        "price", "price_per_m2",                            # target + leakage
        "property_url", "address", "city", "postal_code",  # identifiers / high-cardinality
        "coord_swapped", "coord_suspect",
        "price_suspect", "area_suspect", "year_suspect",
    ]

    def __init__(self, raw_df, clean_df):
        self.raw = raw_df
        self.df = clean_df

    def which_columns_to_drop(self):
        """Q1 — identify columns to delete and visualise why.

        Reasons detected on the RAW data: constant (1 value), duplicate
        (identical to another column), or mostly empty (>75% missing).
        """
        raw = self.raw
        reasons = []

        for c in raw.columns:                                   # constant columns
            if raw[c].nunique(dropna=False) <= 1:
                reasons.append((c, "constant (1 unique value)"))

        seen = {}                                               # duplicate columns
        for c in raw.columns:
            key = tuple(raw[c].fillna("∅").astype(str))
            if key in seen:
                reasons.append((c, f"duplicate of '{seen[key]}'"))
            else:
                seen[key] = c

        miss = raw.isna().mean()                                # mostly-empty columns
        for c in miss[miss > 0.75].index:
            reasons.append((c, f"mostly empty ({miss[c]*100:.0f}% missing)"))

        report = pd.DataFrame(reasons, columns=["column", "reason"])

        fig, axes = plt.subplots(1, 2, figsize=(18, 9))
        fig.suptitle("Q1 — Which variables to delete, and why", fontsize=16, fontweight="bold")

        miss_sorted = (miss * 100).sort_values(ascending=True)
        colors = ["#C44E52" if v > 75 else "#4C72B0" for v in miss_sorted.values]
        miss_sorted.plot(kind="barh", ax=axes[0], color=colors)
        axes[0].axvline(75, color="#C44E52", linestyle="--", linewidth=1)
        axes[0].set_title("Missing values per column (red dashed = 75%)")
        axes[0].set_xlabel("% missing")

        nun = raw.nunique(dropna=False).sort_values()
        colors2 = ["#C44E52" if v <= 1 else "#55A868" for v in nun.values]
        nun.plot(kind="barh", ax=axes[1], color=colors2, logx=True)
        axes[1].set_title("Unique values per column (log scale; red = constant)")
        axes[1].set_xlabel("# unique values")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
        return report

    def _build_feature_matrix(self, data):
        """Encode features for a tree model: numeric as-is, categoricals as codes.

        NOTE: nominal categories are label-encoded for simplicity, which keeps
        ONE importance per variable — convenient to answer "which variables
        matter". A tree model handles this fine for ranking.
        """
        feats = [c for c in data.columns if c not in self.NON_FEATURES]
        X = pd.DataFrame(index=data.index)
        for c in feats:
            s = data[c]
            if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
                X[c] = s.astype("category").cat.codes          # NaN -> -1
            else:
                X[c] = pd.to_numeric(s, errors="coerce")
        X = X.fillna(X.median(numeric_only=True))
        return X

    def top_price_drivers(self, n=5):
        """Q2 — rank the variables that most drive the price (RandomForest)."""
        df = self.df
        data = df.dropna(subset=["price"]).copy()        
        X = self._build_feature_matrix(data)
        y = data["price"]

        model = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0)
        model.fit(X, y)

        importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
        top = importance.head(n)

        plt.figure(figsize=(11, 9))
        colors = ["#C44E52" if i < n else "#9aa0b3" for i in range(len(importance))]
        importance.sort_values().plot(kind="barh", color=colors[::-1])
        plt.title(f"Q2 — Variables driving the price (top {n} in red)", fontsize=15, fontweight="bold")
        plt.xlabel("Importance (RandomForest)")
        plt.tight_layout()
        plt.show()
        return top

### Q1 — Which variables would you delete, and why?

`which_columns_to_drop()` detects three deletion reasons on the **raw** data and plots them.

In [ ]:
qa = MeetingAnswers(cleaner.raw, df)
report = qa.which_columns_to_drop()
report

**Answer.** Three deletions, three distinct reasons:

- **`price_type` — constant.** A single value (`sell`) → zero information (red bar, right panel).
- **`property_id` / `property_url` — duplicate.** Two identical columns → keep one, drop the other.
- **`garden_area_m2` — mostly empty (78% missing).** Weak, unreliable signal (red bar above the 75% line).
  `kitchen_equipped` (74%) and `floor_number` (71%) are borderline candidates of the same kind.

In short: we drop for **constant** (no info), **duplicate** (redundant), or **mostly empty** (weak signal).

### Q2 — The five variables that most influence the price

We train a **RandomForest** to predict `price` and read its feature importances, after excluding the
target, the leakage feature `price_per_m2`, identifiers, and the flagged suspect rows.

In [ ]:
top5 = qa.top_price_drivers(n=5)
top5.round(3)

**Answer — top 5 drivers:**

1. **`living_area_m2`** — by far the strongest: bigger living space = higher price.
2. **`latitude`** — a location proxy: the north/south gradient (Flanders vs Wallonia, urban vs rural).
3. **`total_area_m2`** — total surface (land + building): more space = more value.
4. **`bedrooms`** — number of bedrooms, a direct proxy for usable size.
5. **`epc_score`** — energy rating: an A property sells better than a G.

The story: **size + location + energy** explain most of the price. `bathrooms`, `km_from_nearby_city`
and `longitude` follow just behind, reinforcing the same two themes.

## 5. Summary

- The data is cleaned **non-destructively**: only indisputable errors are corrected, the rest is flagged.
- **Q1** — drop `price_type` (constant), one of `property_id`/`property_url` (duplicate), and very empty
  columns such as `garden_area_m2`.
- **Q2** — the price is driven mainly by **size** (`living_area_m2`, `total_area_m2`, `bedrooms`),
  **location** (`latitude`), and **energy class** (`epc_score`).
